# IOT_Based_Crop_Disease_Detection_System_For_Smart_Agriculture


### Import packages

In [ ]:
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import torch
import torchvision
import torch.nn as nn
from torchvision import models
import torch.optim as optim


from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader

### Check GPU / CPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Import Datast

### Mount Google Drive

In [ ]:
from google.colab import drive # type: ignore
drive.mount('/content/drive')

### Load Train , Valid , Test 

In [ ]:
train_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/valid",
    transform=valid_transform
)

test_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/test",
    transform=valid_transform
)

### train_transform 
- The training transform takes a raw JPG/JPEG RGB image, applies random data augmentation (crop, flip, and rotation), converts it into a normalized float32 PyTorch tensor of shape (3, 224, 224), and prepares it as input for EfficientNet-B0 during model training.

In [ ]:
# Define preprocessing and augmentation

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.9, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
valid_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = valid_transform

In [ ]:
# Create ImageFolder dataset

train_dir =  "/content/drive/MyDrive/dataset/Crop_Disease_Split/train"

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

#  Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Print information about the dataset
print("Number of images:", len(train_dataset))
print("Classes:", train_dataset.classes)
print("Class to index:", train_dataset.class_to_idx)

#  Load one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

### Load the pretrained EfficientNet-B0 model

In [ ]:

# Number of classes
NUM_CLASSES = 19

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Replace the final classifier
model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

# Select device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model = model.to(device)

# Print model information
print(model)
print("\nOutput classes:", model.classifier[1].out_features)
print("Device:", device)

In [ ]:
# Loss Function

criterion = nn.CrossEntropyLoss()

In [ ]:
# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3
)

In [ ]:
# Number of training epochs
num_epochs = 20

for epoch in range(num_epochs):

    # Set model to training mode
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    # Loop through every batch
    for images, labels in train_loader:

        # Move data to GPU/CPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward Pass
        outputs = model(images)

        # Calculate Loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update Model Weights
        optimizer.step()

        # Statistics
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    # Update learning rate scheduler
    scheduler.step(epoch_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )